# Sampling Methods -- Technical Reference

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Simple random sampling | Baseline, homogeneous population | §1 |
| Stratified sampling | Subgroups matter, rare strata | §2 |
| Systematic sampling | Ordered / sequential data | §3 |
| Cluster sampling | Geographically dispersed, costly enumeration | §4 |
| Bootstrap resampling | Unknown distribution, any statistic | §5 |
| Jackknife resampling | Bias correction, smooth statistics | §6 |
| Weighted sampling | Non-response, oversampling correction | §7 |
| Reservoir sampling | Streaming / out-of-memory data | §8 |

---
## When to Use

| Signal | Method to reach for |
| :--- | :--- |
| Population is homogeneous, no subgroup structure | Simple Random Sampling (§1) |
| Subgroups must all be represented, especially rare ones | Stratified Sampling (§2) |
| Data is ordered and drawing every k-th unit is practical | Systematic Sampling (§3) |
| Individuals are expensive to reach; natural clusters exist | Cluster Sampling (§4) |
| Need a CI for a non-standard statistic without distributional assumptions | Bootstrap (§5) |
| Need bias correction for a smooth statistic with limited data | Jackknife (§6) |
| Observations have unequal selection probability | Weighted Sampling (§7) |
| Dataset is too large for memory or arrives as a stream | Reservoir Sampling (§8) |

---
## §1 -- Simple Random Sampling (SRS)

**Intuition:** Every unit in the population has an *equal* probability of being selected -- the "lottery" model. Draw n units with or without replacement. It is the foundation that all other methods refine or depart from. Works well when the population is homogeneous; breaks down when important subgroups are small or rare events must be represented.

**Key properties:**
- Unbiased estimator of the population mean
- Variance decreases as 1/n
- Without replacement is slightly more efficient than with replacement for finite populations

**Confidence Interval -- Normal approximation:**

$$\bar{x} \pm t_{\alpha/2,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

where s is the sample standard deviation. Uses the t-distribution so it is valid for any sample size.

In [ ]:
import numpy as np
import pandas as pd
import math

# ── Pure NumPy/pandas statistical helpers ────────────────────────────────────
def _norm_ppf(p):
    """Inverse normal CDF (Beasley-Springer-Moro rational approximation)."""
    p = float(np.clip(p, 1e-15, 1 - 1e-15))
    sign, q = (1.0, 1.0 - p) if p >= 0.5 else (-1.0, p)
    r = math.sqrt(-2.0 * math.log(q))
    num = 2.515517 + 0.802853*r + 0.010328*r*r
    den = 1.0 + 1.432788*r + 0.189269*r*r + 0.001308*r*r*r
    return sign * (r - num / den)

def _norm_cdf(z):
    """Standard normal CDF via math.erfc (stdlib only)."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def _t_cdf(t, df):
    """CDF of t-distribution at t >= 0."""
    return 1.0 - 0.5 * _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def _t_pdf(t, df):
    log_c = math.lgamma((df+1)/2) - math.lgamma(df/2) - 0.5*math.log(df*math.pi)
    return math.exp(log_c - (df+1)/2 * math.log(1 + t*t/df))

def t_ppf(p, df):
    """
    Inverse CDF of the t-distribution (pure NumPy -- no scipy).
    Cornish-Fisher start (df>=5) or bisection (df<5), then Newton-Raphson.
    Accurate to < 0.002% vs scipy for all df >= 1.
    """
    p = float(p); df = float(df)
    if df >= 1e6: return _norm_ppf(p)
    if p < 0.5:   return -t_ppf(1.0 - p, df)
    if p >= 1.0:  return math.inf
    if df >= 5:
        z0 = _norm_ppf(p)                                              # Cornish-Fisher (z0-based)
        z  = (z0
              + (z0**3 + z0) / (4*df)
              + (5*z0**5 + 16*z0**3 + 3*z0) / (96*df**2)
              + (3*z0**7 + 19*z0**5 + 17*z0**3 - 15*z0) / (384*df**3))
    else:
        lo, hi = 0.0, 1000.0                                           # bisection for df < 5
        for _ in range(60):
            mid = (lo + hi) / 2
            if _t_cdf(mid, df) < p: lo = mid
            else: hi = mid
        z = (lo + hi) / 2
    for _ in range(20):                                                # Newton-Raphson polish
        cdf = _t_cdf(abs(z), df); err = cdf - p
        pdf = _t_pdf(abs(z), df)
        if pdf < 1e-300: break
        z -= err / pdf
        if abs(err) < 1e-14: break
    return z

def sem(x):
    """Standard error of the mean."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def t_interval(confidence, df, loc, scale):
    """Two-sided t confidence interval."""
    tc = t_ppf(1 - (1 - confidence) / 2, df)
    return (loc - tc * scale, loc + tc * scale)

def bca_bootstrap_ci(data, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """
    Bias-Corrected and Accelerated (BCa) bootstrap CI (pure NumPy).
    data    : 1-D NumPy array
    stat_fn : function(array) or function(array, axis=) for vectorised use
    """
    rng = np.random.default_rng(seed)
    n   = len(data)
    obs = float(stat_fn(data))
    idx        = rng.integers(0, n, size=(B, n))
    boot_stats = stat_fn(data[idx], axis=1)
    # Bias-correction z0
    prop = float(np.clip(np.mean(boot_stats < obs), 1e-6, 1 - 1e-6))
    z0   = _norm_ppf(prop)
    # Acceleration a via jackknife
    jk   = np.array([float(stat_fn(np.delete(data, i))) for i in range(n)])
    jk_m = jk.mean()
    num  = np.sum((jk_m - jk) ** 3)
    den  = 6.0 * (np.sum((jk_m - jk) ** 2) ** 1.5)
    a    = float(num / den) if den != 0 else 0.0
    alpha = 1 - confidence
    def adj_pct(tail_p):
        z_a = _norm_ppf(tail_p)
        return 100.0 * _norm_cdf(z0 + (z0 + z_a) / (1.0 - a * (z0 + z_a)))
    return (float(np.percentile(boot_stats, adj_pct(alpha / 2))),
            float(np.percentile(boot_stats, adj_pct(1 - alpha / 2))))

rng = np.random.default_rng(42)

# Simulate population
population = rng.normal(loc=50, scale=10, size=10_000)
df_pop = pd.DataFrame({'value': population})

# SRS without replacement -- pandas
n = 200
sample    = df_pop.sample(n=n, replace=False, random_state=42)
sample_wr = df_pop.sample(n=n, replace=True,  random_state=42)

# SRS using NumPy
sample_np    = rng.choice(population, size=n, replace=False)
sample_np_wr = rng.choice(population, size=n, replace=True)

# Confidence Interval
x     = sample['value'].values
x_bar = x.mean()
se_   = sem(x)                                                         # s / sqrt(n)
ci    = t_interval(0.95, df=n - 1, loc=x_bar, scale=se_)

print(f"Sample mean : {x_bar:.3f}")
print(f"Std error   : {se_:.3f}")
print(f"95% CI      : ({ci[0]:.3f}, {ci[1]:.3f})")
print(f"True mean   : {population.mean():.3f}")

**With-replacement vs. Without-replacement:**

| | Variance | Suitable for | Notes |
| :--- | :--- | :--- | :--- |
| Without replacement | Lower (finite-population correction) | Survey sampling | Default choice |
| With replacement | Slightly higher | Bootstrap (§5) | Required for resampling methods |

**Common mistakes:**
- Forgetting `random_state` / `rng` seed -- results are not reproducible
- Applying SRS to heavily skewed data or data with rare subgroups -- use stratified sampling instead
- Using `replace=True` (with replacement) when estimating a proportion from a finite population -- inflates variance

---
## §2 -- Stratified Sampling

**Intuition:** Divide the population into non-overlapping, exhaustive subgroups (strata) -- e.g. age bands, device type, region -- then independently draw a random sample within each stratum. The key insight is that variance within a stratum is usually lower than variance across the whole population, so combining stratum-level estimates gives a more precise overall estimate than SRS of the same total size. Particularly valuable when some strata are small but analytically important (e.g. a minority user segment in an A/B test).

**Two allocation strategies:**
- **Proportional:** sample each stratum at the same fraction `frac` -- preserves population proportions in the sample
- **Disproportional (Optimal/Neyman):** allocate more to strata with higher variance or lower sampling cost

**Confidence Interval -- Weighted combination of stratum variances:**

$$\bar{x}_{st} = \sum_h W_h \bar{x}_h, \quad \text{Var}(\bar{x}_{st}) = \sum_h W_h^2 \frac{s_h^2}{n_h}$$

$$\text{CI} = \bar{x}_{st} \pm t_{\alpha/2} \cdot \sqrt{\text{Var}(\bar{x}_{st})}$$

where $W_h = N_h / N$ is the stratum weight, $s_h^2$ is the within-stratum variance, and $n_h$ is the stratum sample size.

In [ ]:
import numpy as np
import pandas as pd
import math

# ── Pure NumPy/pandas statistical helpers ────────────────────────────────────
def _norm_ppf(p):
    """Inverse normal CDF (Beasley-Springer-Moro rational approximation)."""
    p = float(np.clip(p, 1e-15, 1 - 1e-15))
    sign, q = (1.0, 1.0 - p) if p >= 0.5 else (-1.0, p)
    r = math.sqrt(-2.0 * math.log(q))
    num = 2.515517 + 0.802853*r + 0.010328*r*r
    den = 1.0 + 1.432788*r + 0.189269*r*r + 0.001308*r*r*r
    return sign * (r - num / den)

def _norm_cdf(z):
    """Standard normal CDF via math.erfc (stdlib only)."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def _t_cdf(t, df):
    """CDF of t-distribution at t >= 0."""
    return 1.0 - 0.5 * _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def _t_pdf(t, df):
    log_c = math.lgamma((df+1)/2) - math.lgamma(df/2) - 0.5*math.log(df*math.pi)
    return math.exp(log_c - (df+1)/2 * math.log(1 + t*t/df))

def t_ppf(p, df):
    """
    Inverse CDF of the t-distribution (pure NumPy -- no scipy).
    Cornish-Fisher start (df>=5) or bisection (df<5), then Newton-Raphson.
    Accurate to < 0.002% vs scipy for all df >= 1.
    """
    p = float(p); df = float(df)
    if df >= 1e6: return _norm_ppf(p)
    if p < 0.5:   return -t_ppf(1.0 - p, df)
    if p >= 1.0:  return math.inf
    if df >= 5:
        z0 = _norm_ppf(p)                                              # Cornish-Fisher (z0-based)
        z  = (z0
              + (z0**3 + z0) / (4*df)
              + (5*z0**5 + 16*z0**3 + 3*z0) / (96*df**2)
              + (3*z0**7 + 19*z0**5 + 17*z0**3 - 15*z0) / (384*df**3))
    else:
        lo, hi = 0.0, 1000.0                                           # bisection for df < 5
        for _ in range(60):
            mid = (lo + hi) / 2
            if _t_cdf(mid, df) < p: lo = mid
            else: hi = mid
        z = (lo + hi) / 2
    for _ in range(20):                                                # Newton-Raphson polish
        cdf = _t_cdf(abs(z), df); err = cdf - p
        pdf = _t_pdf(abs(z), df)
        if pdf < 1e-300: break
        z -= err / pdf
        if abs(err) < 1e-14: break
    return z

def sem(x):
    """Standard error of the mean."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def t_interval(confidence, df, loc, scale):
    """Two-sided t confidence interval."""
    tc = t_ppf(1 - (1 - confidence) / 2, df)
    return (loc - tc * scale, loc + tc * scale)

def bca_bootstrap_ci(data, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """
    Bias-Corrected and Accelerated (BCa) bootstrap CI (pure NumPy).
    data    : 1-D NumPy array
    stat_fn : function(array) or function(array, axis=) for vectorised use
    """
    rng = np.random.default_rng(seed)
    n   = len(data)
    obs = float(stat_fn(data))
    idx        = rng.integers(0, n, size=(B, n))
    boot_stats = stat_fn(data[idx], axis=1)
    # Bias-correction z0
    prop = float(np.clip(np.mean(boot_stats < obs), 1e-6, 1 - 1e-6))
    z0   = _norm_ppf(prop)
    # Acceleration a via jackknife
    jk   = np.array([float(stat_fn(np.delete(data, i))) for i in range(n)])
    jk_m = jk.mean()
    num  = np.sum((jk_m - jk) ** 3)
    den  = 6.0 * (np.sum((jk_m - jk) ** 2) ** 1.5)
    a    = float(num / den) if den != 0 else 0.0
    alpha = 1 - confidence
    def adj_pct(tail_p):
        z_a = _norm_ppf(tail_p)
        return 100.0 * _norm_cdf(z0 + (z0 + z_a) / (1.0 - a * (z0 + z_a)))
    return (float(np.percentile(boot_stats, adj_pct(alpha / 2))),
            float(np.percentile(boot_stats, adj_pct(1 - alpha / 2))))

rng = np.random.default_rng(42)

# Simulate population with three strata
N = 10_000
groups = rng.choice(['A', 'B', 'C'], size=N, p=[0.6, 0.3, 0.1])
means  = {'A': 50, 'B': 65, 'C': 80}
values = np.array([rng.normal(means[g], 10) for g in groups])
df_pop = pd.DataFrame({'value': values, 'group': groups})

# Proportional stratified sample (3% from each stratum)
frac = 0.03
strat_parts = []
for g, grp in df_pop.groupby('group'):
    strat_parts.append(grp.sample(frac=frac, random_state=42))
strat = pd.concat(strat_parts).reset_index(drop=True)

print("Stratum sample sizes:")
print(strat['group'].value_counts().sort_index())

# Disproportional: fixed equal sample sizes per stratum
n_per_stratum = {'A': 30, 'B': 30, 'C': 30}
disp_parts = []
for g, grp in df_pop.groupby('group'):
    disp_parts.append(grp.sample(n=n_per_stratum[g], random_state=42))
disp = pd.concat(disp_parts).reset_index(drop=True)

# Confidence Interval
N_total      = len(df_pop)
group_counts = df_pop['group'].value_counts()
group_stats  = strat.groupby('group').agg(
    n    = ('value', 'count'),
    mean = ('value', 'mean'),
    var  = ('value', 'var')
)
group_stats['N_h'] = group_counts
group_stats['W_h'] = group_stats['N_h'] / N_total

strat_mean = (group_stats['W_h'] * group_stats['mean']).sum()
strat_var  = (group_stats['W_h']**2 * group_stats['var'] / group_stats['n']).sum()
strat_se   = np.sqrt(strat_var)
t_crit     = t_ppf(0.975, df=len(strat) - 1)
ci         = (strat_mean - t_crit * strat_se, strat_mean + t_crit * strat_se)

print(f"\nStratified mean : {strat_mean:.3f}")
print(f"95% CI          : ({ci[0]:.3f}, {ci[1]:.3f})")
print(f"True mean       : {df_pop['value'].mean():.3f}")
print()
print(group_stats[['n', 'mean', 'W_h']])

**Stratified vs. Simple Random Sampling:**

| | Variance | Requires | Best when |
| :--- | :--- | :--- | :--- |
| SRS | Higher (if strata differ) | Nothing | Homogeneous population |
| Stratified | Lower (within-stratum variance) | Known stratum membership | Clear subgroup structure |

**Common mistakes:**
- Using too many strata -- each stratum needs at least 2 observations to estimate variance
- Forgetting to reset the index after `pd.concat` -- duplicate indices cause silent bugs in downstream `.loc` calls
- Using disproportional allocation without re-weighting in the CI formula -- naive mean of the sample will be biased

---
## §3 -- Systematic Sampling

**Intuition:** Sort the population (or use its natural order), choose a random starting point r in [1, k], then select every k-th unit: r, r+k, r+2k, ... This is simpler to execute than SRS in the field (no need to look up random IDs) and often gives similar or better precision when the population is randomly ordered. The danger: if the data has a periodic pattern with the same period as k, the sample can be catastrophically biased -- every selected unit falls on the same phase of the cycle.

**Step size:** k = N / n (rounded to the nearest integer).

**Confidence Interval:** Treated as approximately SRS when there is no periodicity:

$$\text{CI} \approx \bar{x} \pm t_{\alpha/2,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

Note: this underestimates variance when the population has autocorrelation. Use `pd.Series.autocorr()` to check before applying this approximation.

In [ ]:
import numpy as np
import pandas as pd
import math

# ── Pure NumPy/pandas statistical helpers ────────────────────────────────────
def _norm_ppf(p):
    """Inverse normal CDF (Beasley-Springer-Moro rational approximation)."""
    p = float(np.clip(p, 1e-15, 1 - 1e-15))
    sign, q = (1.0, 1.0 - p) if p >= 0.5 else (-1.0, p)
    r = math.sqrt(-2.0 * math.log(q))
    num = 2.515517 + 0.802853*r + 0.010328*r*r
    den = 1.0 + 1.432788*r + 0.189269*r*r + 0.001308*r*r*r
    return sign * (r - num / den)

def _norm_cdf(z):
    """Standard normal CDF via math.erfc (stdlib only)."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def _t_cdf(t, df):
    """CDF of t-distribution at t >= 0."""
    return 1.0 - 0.5 * _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def _t_pdf(t, df):
    log_c = math.lgamma((df+1)/2) - math.lgamma(df/2) - 0.5*math.log(df*math.pi)
    return math.exp(log_c - (df+1)/2 * math.log(1 + t*t/df))

def t_ppf(p, df):
    """
    Inverse CDF of the t-distribution (pure NumPy -- no scipy).
    Cornish-Fisher start (df>=5) or bisection (df<5), then Newton-Raphson.
    Accurate to < 0.002% vs scipy for all df >= 1.
    """
    p = float(p); df = float(df)
    if df >= 1e6: return _norm_ppf(p)
    if p < 0.5:   return -t_ppf(1.0 - p, df)
    if p >= 1.0:  return math.inf
    if df >= 5:
        z0 = _norm_ppf(p)                                              # Cornish-Fisher (z0-based)
        z  = (z0
              + (z0**3 + z0) / (4*df)
              + (5*z0**5 + 16*z0**3 + 3*z0) / (96*df**2)
              + (3*z0**7 + 19*z0**5 + 17*z0**3 - 15*z0) / (384*df**3))
    else:
        lo, hi = 0.0, 1000.0                                           # bisection for df < 5
        for _ in range(60):
            mid = (lo + hi) / 2
            if _t_cdf(mid, df) < p: lo = mid
            else: hi = mid
        z = (lo + hi) / 2
    for _ in range(20):                                                # Newton-Raphson polish
        cdf = _t_cdf(abs(z), df); err = cdf - p
        pdf = _t_pdf(abs(z), df)
        if pdf < 1e-300: break
        z -= err / pdf
        if abs(err) < 1e-14: break
    return z

def sem(x):
    """Standard error of the mean."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def t_interval(confidence, df, loc, scale):
    """Two-sided t confidence interval."""
    tc = t_ppf(1 - (1 - confidence) / 2, df)
    return (loc - tc * scale, loc + tc * scale)

def bca_bootstrap_ci(data, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """
    Bias-Corrected and Accelerated (BCa) bootstrap CI (pure NumPy).
    data    : 1-D NumPy array
    stat_fn : function(array) or function(array, axis=) for vectorised use
    """
    rng = np.random.default_rng(seed)
    n   = len(data)
    obs = float(stat_fn(data))
    idx        = rng.integers(0, n, size=(B, n))
    boot_stats = stat_fn(data[idx], axis=1)
    # Bias-correction z0
    prop = float(np.clip(np.mean(boot_stats < obs), 1e-6, 1 - 1e-6))
    z0   = _norm_ppf(prop)
    # Acceleration a via jackknife
    jk   = np.array([float(stat_fn(np.delete(data, i))) for i in range(n)])
    jk_m = jk.mean()
    num  = np.sum((jk_m - jk) ** 3)
    den  = 6.0 * (np.sum((jk_m - jk) ** 2) ** 1.5)
    a    = float(num / den) if den != 0 else 0.0
    alpha = 1 - confidence
    def adj_pct(tail_p):
        z_a = _norm_ppf(tail_p)
        return 100.0 * _norm_cdf(z0 + (z0 + z_a) / (1.0 - a * (z0 + z_a)))
    return (float(np.percentile(boot_stats, adj_pct(alpha / 2))),
            float(np.percentile(boot_stats, adj_pct(1 - alpha / 2))))

rng = np.random.default_rng(42)

N = 10_000
n = 200
population = rng.normal(loc=50, scale=10, size=N)
df_pop = pd.DataFrame({'value': population})

# Systematic sampling
k     = N // n                                                         # step size = 50
start = int(rng.integers(0, k))                                        # random start in [0, k)
indices = np.arange(start, N, step=k)[:n]

sample_sys = df_pop.iloc[indices]                                      # pandas
sample_np  = population[indices]                                       # NumPy

print(f"Step size k  : {k}")
print(f"Random start : {start}")
print(f"Sample size  : {len(sample_sys)}")

# Confidence Interval (SRS approximation)
x  = sample_sys['value'].values
ci = t_interval(0.95, df=n - 1, loc=x.mean(), scale=sem(x))

print(f"\nSample mean : {x.mean():.3f}")
print(f"95% CI      : ({ci[0]:.3f}, {ci[1]:.3f})")
print(f"True mean   : {population.mean():.3f}")

# Danger: periodic data aligned with step k
periodic      = np.sin(2 * np.pi * np.arange(N) / k) * 20 + 50
periodic_samp = periodic[indices]
print(f"\nPeriodic data true mean   : {periodic.mean():.3f}")
print(f"Periodic data sample mean : {periodic_samp.mean():.3f}  <- biased when phase aligns")

**Common mistakes:**
- Not randomizing the starting point -- the sample is no longer probability-based
- Using systematic sampling on data sorted by the outcome of interest (e.g. revenue-sorted sales data) -- can bias estimates
- Assuming the SRS CI is valid without checking for autocorrelation; use `pd.Series.autocorr()` first

---
## §4 -- Cluster Sampling

**Intuition:** When it is impractical or too costly to list and sample individual units, sample entire naturally-occurring groups (clusters) instead. One-stage cluster sampling takes all units in selected clusters; two-stage subsamples within the selected clusters. Common in field surveys (schools, hospitals, geographic areas) where visiting each individual unit is expensive.

**The variance trade-off:** Units within a cluster tend to be similar (intra-cluster correlation rho > 0), which *increases* variance compared to SRS for the same total n. Cluster sampling trades statistical efficiency for logistical/cost efficiency -- the opposite of stratified sampling.

**Confidence Interval -- Between-cluster variance:**

Compute the statistic within each selected cluster, then treat those cluster-level estimates as a small SRS:

$$\bar{x}_{cl} = \frac{1}{m} \sum_{i=1}^{m} \bar{x}_i, \quad SE = \frac{s_{\bar{x}}}{\sqrt{m}}$$

$$\text{CI} = \bar{x}_{cl} \pm t_{\alpha/2,\, m-1} \cdot SE$$

where m is the number of selected clusters.

In [ ]:
import numpy as np
import pandas as pd
import math

# ── Pure NumPy/pandas statistical helpers ────────────────────────────────────
def _norm_ppf(p):
    """Inverse normal CDF (Beasley-Springer-Moro rational approximation)."""
    p = float(np.clip(p, 1e-15, 1 - 1e-15))
    sign, q = (1.0, 1.0 - p) if p >= 0.5 else (-1.0, p)
    r = math.sqrt(-2.0 * math.log(q))
    num = 2.515517 + 0.802853*r + 0.010328*r*r
    den = 1.0 + 1.432788*r + 0.189269*r*r + 0.001308*r*r*r
    return sign * (r - num / den)

def _norm_cdf(z):
    """Standard normal CDF via math.erfc (stdlib only)."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def _t_cdf(t, df):
    """CDF of t-distribution at t >= 0."""
    return 1.0 - 0.5 * _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def _t_pdf(t, df):
    log_c = math.lgamma((df+1)/2) - math.lgamma(df/2) - 0.5*math.log(df*math.pi)
    return math.exp(log_c - (df+1)/2 * math.log(1 + t*t/df))

def t_ppf(p, df):
    """
    Inverse CDF of the t-distribution (pure NumPy -- no scipy).
    Cornish-Fisher start (df>=5) or bisection (df<5), then Newton-Raphson.
    Accurate to < 0.002% vs scipy for all df >= 1.
    """
    p = float(p); df = float(df)
    if df >= 1e6: return _norm_ppf(p)
    if p < 0.5:   return -t_ppf(1.0 - p, df)
    if p >= 1.0:  return math.inf
    if df >= 5:
        z0 = _norm_ppf(p)                                              # Cornish-Fisher (z0-based)
        z  = (z0
              + (z0**3 + z0) / (4*df)
              + (5*z0**5 + 16*z0**3 + 3*z0) / (96*df**2)
              + (3*z0**7 + 19*z0**5 + 17*z0**3 - 15*z0) / (384*df**3))
    else:
        lo, hi = 0.0, 1000.0                                           # bisection for df < 5
        for _ in range(60):
            mid = (lo + hi) / 2
            if _t_cdf(mid, df) < p: lo = mid
            else: hi = mid
        z = (lo + hi) / 2
    for _ in range(20):                                                # Newton-Raphson polish
        cdf = _t_cdf(abs(z), df); err = cdf - p
        pdf = _t_pdf(abs(z), df)
        if pdf < 1e-300: break
        z -= err / pdf
        if abs(err) < 1e-14: break
    return z

def sem(x):
    """Standard error of the mean."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def t_interval(confidence, df, loc, scale):
    """Two-sided t confidence interval."""
    tc = t_ppf(1 - (1 - confidence) / 2, df)
    return (loc - tc * scale, loc + tc * scale)

def bca_bootstrap_ci(data, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """
    Bias-Corrected and Accelerated (BCa) bootstrap CI (pure NumPy).
    data    : 1-D NumPy array
    stat_fn : function(array) or function(array, axis=) for vectorised use
    """
    rng = np.random.default_rng(seed)
    n   = len(data)
    obs = float(stat_fn(data))
    idx        = rng.integers(0, n, size=(B, n))
    boot_stats = stat_fn(data[idx], axis=1)
    # Bias-correction z0
    prop = float(np.clip(np.mean(boot_stats < obs), 1e-6, 1 - 1e-6))
    z0   = _norm_ppf(prop)
    # Acceleration a via jackknife
    jk   = np.array([float(stat_fn(np.delete(data, i))) for i in range(n)])
    jk_m = jk.mean()
    num  = np.sum((jk_m - jk) ** 3)
    den  = 6.0 * (np.sum((jk_m - jk) ** 2) ** 1.5)
    a    = float(num / den) if den != 0 else 0.0
    alpha = 1 - confidence
    def adj_pct(tail_p):
        z_a = _norm_ppf(tail_p)
        return 100.0 * _norm_cdf(z0 + (z0 + z_a) / (1.0 - a * (z0 + z_a)))
    return (float(np.percentile(boot_stats, adj_pct(alpha / 2))),
            float(np.percentile(boot_stats, adj_pct(1 - alpha / 2))))

rng = np.random.default_rng(42)

# Simulate 20 clusters with variable size and different means
n_clusters    = 20
cluster_means = rng.normal(50, 8, size=n_clusters)
rows = []
for cid, cmean in enumerate(cluster_means):
    size = int(rng.integers(40, 80))
    vals = rng.normal(cmean, 5, size=size)
    rows.append(pd.DataFrame({'value': vals, 'cluster_id': cid}))
df_pop = pd.concat(rows, ignore_index=True)
print(f"Population: {len(df_pop)} units across {n_clusters} clusters")

# One-stage cluster sampling
m        = 6
selected = rng.choice(n_clusters, size=m, replace=False)
cluster_sample = df_pop[df_pop['cluster_id'].isin(selected)].copy()
print(f"\nSelected clusters : {[int(c) for c in sorted(selected)]}")
print(f"One-stage sample  : {len(cluster_sample)} units")

# Two-stage: subsample 50% within each selected cluster
two_stage_parts = []
for cid in selected:
    grp = df_pop[df_pop['cluster_id'] == cid]
    two_stage_parts.append(grp.sample(frac=0.5, random_state=42))
two_stage = pd.concat(two_stage_parts, ignore_index=True)
print(f"Two-stage sample  : {len(two_stage)} units")

# Confidence Interval using between-cluster variance
cluster_means_sample = cluster_sample.groupby('cluster_id')['value'].mean()
c_bar = float(cluster_means_sample.mean())
c_se  = sem(cluster_means_sample.values)
ci    = t_interval(0.95, df=m - 1, loc=c_bar, scale=c_se)

print(f"\nCluster-based mean : {c_bar:.3f}")
print(f"95% CI             : ({ci[0]:.3f}, {ci[1]:.3f})")
print(f"True mean          : {df_pop['value'].mean():.3f}")
print(f"\nNote: CI is wide because it is based on only m={m} cluster-level estimates")

**Cluster vs. Stratified Sampling:**

| | Variance vs SRS | Cost vs SRS | Within-group homogeneity | Use when |
| :--- | :--- | :--- | :--- | :--- |
| Stratified | Lower | Similar | Low (strata are heterogeneous) | Precision is the priority |
| Cluster | Higher | Lower | High (clusters are homogeneous) | Cost / logistics is the constraint |

**Common mistakes:**
- Confusing cluster and stratified sampling -- they have *opposite* effects on variance
- Using m = 2 or 3 clusters -- the t-distribution with 1-2 degrees of freedom gives extremely wide CIs; aim for m >= 5
- Ignoring intra-cluster correlation (ICC) -- high ICC requires more clusters to achieve the same precision as SRS

---
## §5 -- Bootstrap Resampling

**Intuition:** The bootstrap treats your observed sample as a proxy for the population. By repeatedly drawing samples *with replacement* of size n from your data, you simulate the sampling distribution of any statistic without assuming any distributional form. The spread of the bootstrap distribution estimates the standard error; its quantiles give confidence intervals. Works for means, medians, ratios, regression coefficients, AUC, and virtually any other statistic.

**Algorithm:**
1. Draw B samples of size n with replacement from the original data
2. Compute the statistic on each resample to get B bootstrap estimates
3. Use the distribution of those estimates to compute SE and CI

**Three CI methods:**

| Method | Formula | Best when |
| :--- | :--- | :--- |
| Percentile | [q_2.5, q_97.5] of bootstrap stats | Distribution is symmetric and unbiased |
| Basic (pivot) | [2*theta_hat - q_97.5, 2*theta_hat - q_2.5] | Corrects for bias in the bootstrap distribution |
| BCa (Bias-Corrected & Accelerated) | Adjusts for bias and skewness | Best general-purpose choice |

In [ ]:
import numpy as np
import pandas as pd
import math

# ── Pure NumPy/pandas statistical helpers ────────────────────────────────────
def _norm_ppf(p):
    """Inverse normal CDF (Beasley-Springer-Moro rational approximation)."""
    p = float(np.clip(p, 1e-15, 1 - 1e-15))
    sign, q = (1.0, 1.0 - p) if p >= 0.5 else (-1.0, p)
    r = math.sqrt(-2.0 * math.log(q))
    num = 2.515517 + 0.802853*r + 0.010328*r*r
    den = 1.0 + 1.432788*r + 0.189269*r*r + 0.001308*r*r*r
    return sign * (r - num / den)

def _norm_cdf(z):
    """Standard normal CDF via math.erfc (stdlib only)."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def _t_cdf(t, df):
    """CDF of t-distribution at t >= 0."""
    return 1.0 - 0.5 * _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def _t_pdf(t, df):
    log_c = math.lgamma((df+1)/2) - math.lgamma(df/2) - 0.5*math.log(df*math.pi)
    return math.exp(log_c - (df+1)/2 * math.log(1 + t*t/df))

def t_ppf(p, df):
    """
    Inverse CDF of the t-distribution (pure NumPy -- no scipy).
    Cornish-Fisher start (df>=5) or bisection (df<5), then Newton-Raphson.
    Accurate to < 0.002% vs scipy for all df >= 1.
    """
    p = float(p); df = float(df)
    if df >= 1e6: return _norm_ppf(p)
    if p < 0.5:   return -t_ppf(1.0 - p, df)
    if p >= 1.0:  return math.inf
    if df >= 5:
        z0 = _norm_ppf(p)                                              # Cornish-Fisher (z0-based)
        z  = (z0
              + (z0**3 + z0) / (4*df)
              + (5*z0**5 + 16*z0**3 + 3*z0) / (96*df**2)
              + (3*z0**7 + 19*z0**5 + 17*z0**3 - 15*z0) / (384*df**3))
    else:
        lo, hi = 0.0, 1000.0                                           # bisection for df < 5
        for _ in range(60):
            mid = (lo + hi) / 2
            if _t_cdf(mid, df) < p: lo = mid
            else: hi = mid
        z = (lo + hi) / 2
    for _ in range(20):                                                # Newton-Raphson polish
        cdf = _t_cdf(abs(z), df); err = cdf - p
        pdf = _t_pdf(abs(z), df)
        if pdf < 1e-300: break
        z -= err / pdf
        if abs(err) < 1e-14: break
    return z

def sem(x):
    """Standard error of the mean."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def t_interval(confidence, df, loc, scale):
    """Two-sided t confidence interval."""
    tc = t_ppf(1 - (1 - confidence) / 2, df)
    return (loc - tc * scale, loc + tc * scale)

def bca_bootstrap_ci(data, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """
    Bias-Corrected and Accelerated (BCa) bootstrap CI (pure NumPy).
    data    : 1-D NumPy array
    stat_fn : function(array) or function(array, axis=) for vectorised use
    """
    rng = np.random.default_rng(seed)
    n   = len(data)
    obs = float(stat_fn(data))
    idx        = rng.integers(0, n, size=(B, n))
    boot_stats = stat_fn(data[idx], axis=1)
    # Bias-correction z0
    prop = float(np.clip(np.mean(boot_stats < obs), 1e-6, 1 - 1e-6))
    z0   = _norm_ppf(prop)
    # Acceleration a via jackknife
    jk   = np.array([float(stat_fn(np.delete(data, i))) for i in range(n)])
    jk_m = jk.mean()
    num  = np.sum((jk_m - jk) ** 3)
    den  = 6.0 * (np.sum((jk_m - jk) ** 2) ** 1.5)
    a    = float(num / den) if den != 0 else 0.0
    alpha = 1 - confidence
    def adj_pct(tail_p):
        z_a = _norm_ppf(tail_p)
        return 100.0 * _norm_cdf(z0 + (z0 + z_a) / (1.0 - a * (z0 + z_a)))
    return (float(np.percentile(boot_stats, adj_pct(alpha / 2))),
            float(np.percentile(boot_stats, adj_pct(1 - alpha / 2))))

rng  = np.random.default_rng(42)
data = rng.normal(loc=50, scale=10, size=200)
n    = len(data)
B    = 2000

# Bootstrap loop -- pandas
s = pd.Series(data)
boot_means_pd = np.array([
    s.sample(frac=1, replace=True, random_state=i).mean()
    for i in range(B)
])

# Bootstrap -- vectorized NumPy (faster)
idx        = rng.integers(0, n, size=(B, n))
boot_means = data[idx].mean(axis=1)

# CI Method 1: Percentile
theta_hat = data.mean()
pct_ci    = np.percentile(boot_means, [2.5, 97.5])

# CI Method 2: Basic / Pivot
basic_ci = (2 * theta_hat - pct_ci[1], 2 * theta_hat - pct_ci[0])

# CI Method 3: BCa (pure NumPy implementation)
bca_ci = bca_bootstrap_ci(data, stat_fn=np.mean, B=B, seed=42)

print(f"Observed mean      : {theta_hat:.3f}")
print(f"Bootstrap SE       : {boot_means.std(ddof=1):.3f}")
print(f"\n95% CI -- Percentile : ({pct_ci[0]:.3f}, {pct_ci[1]:.3f})")
print(f"95% CI -- Basic      : ({basic_ci[0]:.3f}, {basic_ci[1]:.3f})")
print(f"95% CI -- BCa        : ({bca_ci[0]:.3f}, {bca_ci[1]:.3f})")

**Bootstrap CI methods compared:**

| Method | Handles bias? | Handles skewness? | Complexity | Recommended? |
| :--- | :--- | :--- | :--- | :--- |
| Percentile | No | No | Low | For quick checks only |
| Basic / Pivot | Partially | No | Low | Better than percentile |
| BCa | Yes | Yes | Medium | Best general choice |

**Common mistakes:**
- Using B < 1000 resamples -- CIs are unstable; use B >= 2000 for reliable results
- Bootstrapping time-series or clustered data with simple row resampling -- violates independence; use block bootstrap instead
- Interpreting a wide bootstrap CI as a bug -- it may simply reflect genuine uncertainty in a small or variable dataset

---
## §6 -- Jackknife Resampling

**Intuition:** Leave one observation out at a time, compute the statistic on the remaining n-1 observations, and repeat for every observation. The resulting n leave-one-out estimates reveal how much each individual observation influences the statistic. Their spread is used to estimate variance and bias. The jackknife is computationally cheaper than the bootstrap (exactly n resamples vs. B >= 1000), but only reliable for *smooth* statistics -- those whose value changes continuously as one observation is altered. It fails for the median, quantiles, or any other non-smooth statistic.

**Jackknife bias correction:**

$$\hat{\theta}_{jack} = n\hat{\theta} - (n-1)\bar{\theta}_{(.)}$$

where $\bar{\theta}_{(.)}$ is the mean of the n leave-one-out estimates. Removes O(1/n) bias.

**Confidence Interval:**

$$SE_{jack} = \sqrt{\frac{n-1}{n} \sum_{i=1}^{n} (\hat{\theta}_{(i)} - \bar{\theta}_{(.)})^2}$$

$$\text{CI} = \hat{\theta} \pm t_{\alpha/2,\, n-1} \cdot SE_{jack}$$

In [ ]:
import numpy as np
import pandas as pd
import math

# ── Pure NumPy/pandas statistical helpers ────────────────────────────────────
def _norm_ppf(p):
    """Inverse normal CDF (Beasley-Springer-Moro rational approximation)."""
    p = float(np.clip(p, 1e-15, 1 - 1e-15))
    sign, q = (1.0, 1.0 - p) if p >= 0.5 else (-1.0, p)
    r = math.sqrt(-2.0 * math.log(q))
    num = 2.515517 + 0.802853*r + 0.010328*r*r
    den = 1.0 + 1.432788*r + 0.189269*r*r + 0.001308*r*r*r
    return sign * (r - num / den)

def _norm_cdf(z):
    """Standard normal CDF via math.erfc (stdlib only)."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def _t_cdf(t, df):
    """CDF of t-distribution at t >= 0."""
    return 1.0 - 0.5 * _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def _t_pdf(t, df):
    log_c = math.lgamma((df+1)/2) - math.lgamma(df/2) - 0.5*math.log(df*math.pi)
    return math.exp(log_c - (df+1)/2 * math.log(1 + t*t/df))

def t_ppf(p, df):
    """
    Inverse CDF of the t-distribution (pure NumPy -- no scipy).
    Cornish-Fisher start (df>=5) or bisection (df<5), then Newton-Raphson.
    Accurate to < 0.002% vs scipy for all df >= 1.
    """
    p = float(p); df = float(df)
    if df >= 1e6: return _norm_ppf(p)
    if p < 0.5:   return -t_ppf(1.0 - p, df)
    if p >= 1.0:  return math.inf
    if df >= 5:
        z0 = _norm_ppf(p)                                              # Cornish-Fisher (z0-based)
        z  = (z0
              + (z0**3 + z0) / (4*df)
              + (5*z0**5 + 16*z0**3 + 3*z0) / (96*df**2)
              + (3*z0**7 + 19*z0**5 + 17*z0**3 - 15*z0) / (384*df**3))
    else:
        lo, hi = 0.0, 1000.0                                           # bisection for df < 5
        for _ in range(60):
            mid = (lo + hi) / 2
            if _t_cdf(mid, df) < p: lo = mid
            else: hi = mid
        z = (lo + hi) / 2
    for _ in range(20):                                                # Newton-Raphson polish
        cdf = _t_cdf(abs(z), df); err = cdf - p
        pdf = _t_pdf(abs(z), df)
        if pdf < 1e-300: break
        z -= err / pdf
        if abs(err) < 1e-14: break
    return z

def sem(x):
    """Standard error of the mean."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def t_interval(confidence, df, loc, scale):
    """Two-sided t confidence interval."""
    tc = t_ppf(1 - (1 - confidence) / 2, df)
    return (loc - tc * scale, loc + tc * scale)

def bca_bootstrap_ci(data, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """
    Bias-Corrected and Accelerated (BCa) bootstrap CI (pure NumPy).
    data    : 1-D NumPy array
    stat_fn : function(array) or function(array, axis=) for vectorised use
    """
    rng = np.random.default_rng(seed)
    n   = len(data)
    obs = float(stat_fn(data))
    idx        = rng.integers(0, n, size=(B, n))
    boot_stats = stat_fn(data[idx], axis=1)
    # Bias-correction z0
    prop = float(np.clip(np.mean(boot_stats < obs), 1e-6, 1 - 1e-6))
    z0   = _norm_ppf(prop)
    # Acceleration a via jackknife
    jk   = np.array([float(stat_fn(np.delete(data, i))) for i in range(n)])
    jk_m = jk.mean()
    num  = np.sum((jk_m - jk) ** 3)
    den  = 6.0 * (np.sum((jk_m - jk) ** 2) ** 1.5)
    a    = float(num / den) if den != 0 else 0.0
    alpha = 1 - confidence
    def adj_pct(tail_p):
        z_a = _norm_ppf(tail_p)
        return 100.0 * _norm_cdf(z0 + (z0 + z_a) / (1.0 - a * (z0 + z_a)))
    return (float(np.percentile(boot_stats, adj_pct(alpha / 2))),
            float(np.percentile(boot_stats, adj_pct(1 - alpha / 2))))

rng  = np.random.default_rng(42)
data = rng.normal(loc=50, scale=10, size=200)
n    = len(data)

# Jackknife -- NumPy leave-one-out
jk_stats = np.array([np.mean(np.delete(data, i)) for i in range(n)])

# Jackknife -- pandas
s = pd.Series(data)
jk_stats_pd = np.array([s.drop(index=i).mean() for i in range(n)])

# Bias estimation and correction
theta_hat  = np.mean(data)
theta_bar  = jk_stats.mean()
bias_est   = (n - 1) * (theta_bar - theta_hat)
theta_jack = n * theta_hat - (n - 1) * theta_bar                      # bias-corrected

# Jackknife SE and CI
jk_var = ((n - 1) / n) * np.sum((jk_stats - theta_bar) ** 2)
jk_se  = np.sqrt(jk_var)
t_crit = t_ppf(0.975, df=n - 1)
ci     = (theta_hat - t_crit * jk_se, theta_hat + t_crit * jk_se)

print(f"Original estimate   : {theta_hat:.4f}")
print(f"Jackknife bias est. : {bias_est:.4f}  (small = good)")
print(f"Bias-corrected est. : {theta_jack:.4f}")
print(f"Jackknife SE        : {jk_se:.4f}")
print(f"95% CI              : ({ci[0]:.3f}, {ci[1]:.3f})")

# Demonstrate failure on median
jk_medians = np.array([np.median(np.delete(data, i)) for i in range(n)])
jk_se_med  = np.sqrt(((n-1)/n) * np.sum((jk_medians - jk_medians.mean())**2))
boot_se_med = np.std(
    [np.median(rng.choice(data, n, replace=True)) for _ in range(2000)], ddof=1
)
print(f"\nJackknife SE for median : {jk_se_med:.4f}")
print(f"Bootstrap SE for median : {boot_se_med:.4f}")
print("Use bootstrap for median; jackknife SE is unreliable for non-smooth statistics")

**Jackknife vs. Bootstrap:**

| | Resamples | Works for non-smooth stats? | Bias correction? | Cost |
| :--- | :--- | :--- | :--- | :--- |
| Jackknife | n (exact) | No -- unreliable for median, quantiles | Yes | O(n) |
| Bootstrap | B >= 1000 | Yes | Partial (BCa) | O(B * n) |

**Common mistakes:**
- Using jackknife for the median, maximum, or quantiles -- SE estimates will be incorrect
- Confusing leave-one-out cross-validation (LOOCV) with jackknife -- same structure, different goal: LOOCV estimates prediction error; jackknife estimates variance
- Applying jackknife to very small samples (n < 10) -- high influence of each observation makes bias correction unstable

---
## §7 -- Weighted Sampling

**Intuition:** When observations do not have equal probability of being sampled -- due to deliberate oversampling of rare groups, survey non-response, or study design -- weights restore representativeness. Each observation receives a weight proportional to the inverse of its selection probability: heavier weight means under-represented in the sample and therefore given more "votes" when estimating population quantities. This is the basis of survey-weighted estimation used by national statistics offices worldwide.

**Inverse-probability weighting (IPW):** If stratum h was sampled at rate f_h = n_h / N_h, the weight for an observation in that stratum is w_i = 1 / f_h.

**Confidence Interval -- Weighted mean and variance:**

$$\bar{x}_w = \frac{\sum_i w_i x_i}{\sum_i w_i}, \quad s^2_w = \frac{\sum_i w_i (x_i - \bar{x}_w)^2}{\sum_i w_i}$$

$$SE_w = \sqrt{\frac{s^2_w}{n}}, \quad \text{CI} = \bar{x}_w \pm t_{\alpha/2,\, n-1} \cdot SE_w$$

In [ ]:
import numpy as np
import pandas as pd
import math

# ── Pure NumPy/pandas statistical helpers ────────────────────────────────────
def _norm_ppf(p):
    """Inverse normal CDF (Beasley-Springer-Moro rational approximation)."""
    p = float(np.clip(p, 1e-15, 1 - 1e-15))
    sign, q = (1.0, 1.0 - p) if p >= 0.5 else (-1.0, p)
    r = math.sqrt(-2.0 * math.log(q))
    num = 2.515517 + 0.802853*r + 0.010328*r*r
    den = 1.0 + 1.432788*r + 0.189269*r*r + 0.001308*r*r*r
    return sign * (r - num / den)

def _norm_cdf(z):
    """Standard normal CDF via math.erfc (stdlib only)."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def _t_cdf(t, df):
    """CDF of t-distribution at t >= 0."""
    return 1.0 - 0.5 * _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def _t_pdf(t, df):
    log_c = math.lgamma((df+1)/2) - math.lgamma(df/2) - 0.5*math.log(df*math.pi)
    return math.exp(log_c - (df+1)/2 * math.log(1 + t*t/df))

def t_ppf(p, df):
    """
    Inverse CDF of the t-distribution (pure NumPy -- no scipy).
    Cornish-Fisher start (df>=5) or bisection (df<5), then Newton-Raphson.
    Accurate to < 0.002% vs scipy for all df >= 1.
    """
    p = float(p); df = float(df)
    if df >= 1e6: return _norm_ppf(p)
    if p < 0.5:   return -t_ppf(1.0 - p, df)
    if p >= 1.0:  return math.inf
    if df >= 5:
        z0 = _norm_ppf(p)                                              # Cornish-Fisher (z0-based)
        z  = (z0
              + (z0**3 + z0) / (4*df)
              + (5*z0**5 + 16*z0**3 + 3*z0) / (96*df**2)
              + (3*z0**7 + 19*z0**5 + 17*z0**3 - 15*z0) / (384*df**3))
    else:
        lo, hi = 0.0, 1000.0                                           # bisection for df < 5
        for _ in range(60):
            mid = (lo + hi) / 2
            if _t_cdf(mid, df) < p: lo = mid
            else: hi = mid
        z = (lo + hi) / 2
    for _ in range(20):                                                # Newton-Raphson polish
        cdf = _t_cdf(abs(z), df); err = cdf - p
        pdf = _t_pdf(abs(z), df)
        if pdf < 1e-300: break
        z -= err / pdf
        if abs(err) < 1e-14: break
    return z

def sem(x):
    """Standard error of the mean."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def t_interval(confidence, df, loc, scale):
    """Two-sided t confidence interval."""
    tc = t_ppf(1 - (1 - confidence) / 2, df)
    return (loc - tc * scale, loc + tc * scale)

def bca_bootstrap_ci(data, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """
    Bias-Corrected and Accelerated (BCa) bootstrap CI (pure NumPy).
    data    : 1-D NumPy array
    stat_fn : function(array) or function(array, axis=) for vectorised use
    """
    rng = np.random.default_rng(seed)
    n   = len(data)
    obs = float(stat_fn(data))
    idx        = rng.integers(0, n, size=(B, n))
    boot_stats = stat_fn(data[idx], axis=1)
    # Bias-correction z0
    prop = float(np.clip(np.mean(boot_stats < obs), 1e-6, 1 - 1e-6))
    z0   = _norm_ppf(prop)
    # Acceleration a via jackknife
    jk   = np.array([float(stat_fn(np.delete(data, i))) for i in range(n)])
    jk_m = jk.mean()
    num  = np.sum((jk_m - jk) ** 3)
    den  = 6.0 * (np.sum((jk_m - jk) ** 2) ** 1.5)
    a    = float(num / den) if den != 0 else 0.0
    alpha = 1 - confidence
    def adj_pct(tail_p):
        z_a = _norm_ppf(tail_p)
        return 100.0 * _norm_cdf(z0 + (z0 + z_a) / (1.0 - a * (z0 + z_a)))
    return (float(np.percentile(boot_stats, adj_pct(alpha / 2))),
            float(np.percentile(boot_stats, adj_pct(1 - alpha / 2))))

rng = np.random.default_rng(42)

# Simulate population with three strata sampled at different rates
strata_info = {
    'low':  {'N': 7000, 'mean': 45, 'frac': 0.01},
    'mid':  {'N': 2000, 'mean': 60, 'frac': 0.05},
    'high': {'N': 1000, 'mean': 80, 'frac': 0.10},
}
N_total   = sum(v['N'] for v in strata_info.values())
true_mean = sum(v['N'] * v['mean'] for v in strata_info.values()) / N_total

rows = []
for stratum, info in strata_info.items():
    vals   = rng.normal(info['mean'], 10, size=info['N'])
    n_samp = int(info['N'] * info['frac'])
    samp   = rng.choice(vals, size=n_samp, replace=False)
    weight = 1.0 / info['frac']                                        # IPW = 1 / sampling fraction
    rows.append(pd.DataFrame({'value': samp, 'stratum': stratum, 'weight': weight}))
df = pd.concat(rows, ignore_index=True)

# Weighted sampling -- pandas (normalizes weights automatically)
weighted_samp = df.sample(n=100, weights='weight', random_state=42)

# Weighted sampling -- NumPy (weights must sum to 1 explicitly)
w_norm  = df['weight'].values / df['weight'].values.sum()
idx_np  = rng.choice(len(df), size=100, replace=False, p=w_norm)
np_samp = df.iloc[idx_np]

# Weighted mean and CI
vals_all = df['value'].values
w_all    = df['weight'].values
w_mean   = float(np.average(vals_all, weights=w_all))
w_var    = float(np.average((vals_all - w_mean) ** 2, weights=w_all))
w_se     = math.sqrt(w_var / len(vals_all))
t_crit   = t_ppf(0.975, df=len(vals_all) - 1)
ci       = (w_mean - t_crit * w_se, w_mean + t_crit * w_se)

naive_mean = df['value'].mean()

print(f"True population mean  : {true_mean:.3f}")
print(f"Weighted estimate     : {w_mean:.3f}  <- corrected")
print(f"Naive unweighted mean : {naive_mean:.3f}  <- biased ('high' stratum oversampled)")
print(f"95% CI                : ({ci[0]:.3f}, {ci[1]:.3f})")

**Common mistakes:**
- `df.sample(weights=col)` normalizes automatically; `np.random.choice(p=...)` requires weights that already sum to 1 -- forgetting this raises a `ValueError`
- Double-weighting: constructing weights from sampling fractions and then dividing by them again in the mean formula
- Using population-level weights for a subgroup analysis without rescaling -- weights designed for the full population may give wrong estimates within a subgroup
- Confusing probability weights (IPW, for selection bias) with frequency weights (for pre-aggregated count data)

---
## §8 -- Reservoir Sampling

**Intuition:** When data arrives as a stream -- or when the dataset is too large to load into memory -- you cannot use `df.sample()`. Reservoir sampling (Algorithm R, Vitter 1985) maintains a fixed-size "reservoir" of k items and processes each incoming item exactly once. At step i > k, the new item replaces a random slot in the reservoir with probability k/i. This elegant probability rule ensures that after processing all N items, every item has an equal inclusion probability of k/N -- even though N was never known in advance.

**Algorithm R step-by-step:**
1. Fill the reservoir with the first k items
2. For each subsequent item at position i (i = k+1, k+2, ...): draw j uniformly from [0, i]; if j < k, replace reservoir[j] with the new item
3. After the stream ends, the reservoir is a uniform random sample of size k

**Confidence Interval:** Once the reservoir is collected, treat it as SRS and apply the t-interval from §1:

$$\text{CI} = \bar{x} \pm t_{\alpha/2,\, k-1} \cdot \frac{s}{\sqrt{k}}$$

In [ ]:
import numpy as np
import pandas as pd
import math

# ── Pure NumPy/pandas statistical helpers ────────────────────────────────────
def _norm_ppf(p):
    """Inverse normal CDF (Beasley-Springer-Moro rational approximation)."""
    p = float(np.clip(p, 1e-15, 1 - 1e-15))
    sign, q = (1.0, 1.0 - p) if p >= 0.5 else (-1.0, p)
    r = math.sqrt(-2.0 * math.log(q))
    num = 2.515517 + 0.802853*r + 0.010328*r*r
    den = 1.0 + 1.432788*r + 0.189269*r*r + 0.001308*r*r*r
    return sign * (r - num / den)

def _norm_cdf(z):
    """Standard normal CDF via math.erfc (stdlib only)."""
    return 0.5 * math.erfc(-z / math.sqrt(2))

def _reg_inc_beta(x, a, b):
    """Regularized incomplete beta I(x;a,b) via Lentz continued fraction."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    if x > (a + 1.0) / (a + b + 2.0):
        return 1.0 - _reg_inc_beta(1.0 - x, b, a)
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a*math.log(x) + b*math.log(1.0 - x) - lbeta) / a
    TINY = 1e-30; f = TINY; C = f; D = 0.0
    for m in range(300):
        for s in (0, 1):
            if m == 0 and s == 0: num = 1.0
            elif s == 0: num = m*(b - m)*x / ((a + 2*m - 1)*(a + 2*m))
            else:        num = -(a + m)*(a + b + m)*x / ((a + 2*m)*(a + 2*m + 1))
            D = 1.0 + num*D; D = TINY if abs(D) < TINY else D
            C = 1.0 + num/C; C = TINY if abs(C) < TINY else C
            D = 1.0 / D; delta = C * D; f *= delta
            if abs(delta - 1.0) < 1e-14: return front * f
    return front * f

def _t_cdf(t, df):
    """CDF of t-distribution at t >= 0."""
    return 1.0 - 0.5 * _reg_inc_beta(df / (df + t*t), df / 2.0, 0.5)

def _t_pdf(t, df):
    log_c = math.lgamma((df+1)/2) - math.lgamma(df/2) - 0.5*math.log(df*math.pi)
    return math.exp(log_c - (df+1)/2 * math.log(1 + t*t/df))

def t_ppf(p, df):
    """
    Inverse CDF of the t-distribution (pure NumPy -- no scipy).
    Cornish-Fisher start (df>=5) or bisection (df<5), then Newton-Raphson.
    Accurate to < 0.002% vs scipy for all df >= 1.
    """
    p = float(p); df = float(df)
    if df >= 1e6: return _norm_ppf(p)
    if p < 0.5:   return -t_ppf(1.0 - p, df)
    if p >= 1.0:  return math.inf
    if df >= 5:
        z0 = _norm_ppf(p)                                              # Cornish-Fisher (z0-based)
        z  = (z0
              + (z0**3 + z0) / (4*df)
              + (5*z0**5 + 16*z0**3 + 3*z0) / (96*df**2)
              + (3*z0**7 + 19*z0**5 + 17*z0**3 - 15*z0) / (384*df**3))
    else:
        lo, hi = 0.0, 1000.0                                           # bisection for df < 5
        for _ in range(60):
            mid = (lo + hi) / 2
            if _t_cdf(mid, df) < p: lo = mid
            else: hi = mid
        z = (lo + hi) / 2
    for _ in range(20):                                                # Newton-Raphson polish
        cdf = _t_cdf(abs(z), df); err = cdf - p
        pdf = _t_pdf(abs(z), df)
        if pdf < 1e-300: break
        z -= err / pdf
        if abs(err) < 1e-14: break
    return z

def sem(x):
    """Standard error of the mean."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def t_interval(confidence, df, loc, scale):
    """Two-sided t confidence interval."""
    tc = t_ppf(1 - (1 - confidence) / 2, df)
    return (loc - tc * scale, loc + tc * scale)

def bca_bootstrap_ci(data, stat_fn=np.mean, B=2000, confidence=0.95, seed=42):
    """
    Bias-Corrected and Accelerated (BCa) bootstrap CI (pure NumPy).
    data    : 1-D NumPy array
    stat_fn : function(array) or function(array, axis=) for vectorised use
    """
    rng = np.random.default_rng(seed)
    n   = len(data)
    obs = float(stat_fn(data))
    idx        = rng.integers(0, n, size=(B, n))
    boot_stats = stat_fn(data[idx], axis=1)
    # Bias-correction z0
    prop = float(np.clip(np.mean(boot_stats < obs), 1e-6, 1 - 1e-6))
    z0   = _norm_ppf(prop)
    # Acceleration a via jackknife
    jk   = np.array([float(stat_fn(np.delete(data, i))) for i in range(n)])
    jk_m = jk.mean()
    num  = np.sum((jk_m - jk) ** 3)
    den  = 6.0 * (np.sum((jk_m - jk) ** 2) ** 1.5)
    a    = float(num / den) if den != 0 else 0.0
    alpha = 1 - confidence
    def adj_pct(tail_p):
        z_a = _norm_ppf(tail_p)
        return 100.0 * _norm_cdf(z0 + (z0 + z_a) / (1.0 - a * (z0 + z_a)))
    return (float(np.percentile(boot_stats, adj_pct(alpha / 2))),
            float(np.percentile(boot_stats, adj_pct(1 - alpha / 2))))

def reservoir_sample(stream, k, seed=42):
    """Uniform random sample of size k from any iterator in a single pass."""
    rng_r = np.random.default_rng(seed)
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            j = int(rng_r.integers(0, i + 1))
            if j < k:
                reservoir[j] = item
    return reservoir

def data_stream(n, seed=0):
    rng_s = np.random.default_rng(seed)
    for _ in range(n):
        yield float(rng_s.normal(50, 10))

N = 1_000_000
k = 500

print(f"Processing stream of N={N:,} items, reservoir size k={k} ...")
reservoir = np.array(reservoir_sample(data_stream(N, seed=0), k=k, seed=0))
print(f"Done. First 5 reservoir values: {reservoir[:5].round(2)}")

# Confidence Interval -- treat reservoir as SRS
true_mean = 50.0
res_mean  = float(reservoir.mean())
res_se    = sem(reservoir)
ci        = t_interval(0.95, df=k - 1, loc=res_mean, scale=res_se)

print(f"\nReservoir mean  : {res_mean:.3f}  (true: {true_mean})")
print(f"95% CI          : ({ci[0]:.3f}, {ci[1]:.3f})")
print(f"True mean in CI : {ci[0] <= true_mean <= ci[1]}")
print(f"\nMemory used: ~{reservoir.nbytes / 1024:.1f} KB for {k} items")
print(f"vs. ~{N * 8 / 1e6:.0f} MB if the full stream were stored in memory")

**Reservoir vs. `df.sample()`:**

| Condition | Use |
| :--- | :--- |
| Data fits in memory | `df.sample()` -- simpler and faster |
| Data is a stream or too large for RAM | Reservoir sampling |
| Population size N is unknown in advance | Reservoir sampling -- N never needed |
| Need reproducibility | Both support seeding |

**Common mistakes:**
- Using Python's `random.randint(0, i)` (inclusive) vs. `np.random.integers(0, i+1)` (exclusive upper bound) -- off-by-one errors break the uniform probability guarantee
- Re-seeding the RNG inside the loop -- destroys uniformity of the sample
- Choosing k too small -- the t-interval still applies; small k produces wide CIs regardless of N

---
## §9 -- Practical Decision Guide

```
What is your data situation?
+-- Data fits in memory
|   +-- Population is homogeneous                        -> §1 Simple Random Sampling
|   +-- Subgroups must be represented                    -> §2 Stratified Sampling
|   +-- Data is naturally ordered                        -> §3 Systematic Sampling
|   +-- Individuals are clustered (schools, regions)     -> §4 Cluster Sampling
|   +-- Observations have unequal selection probability  -> §7 Weighted Sampling
|
+-- Data is a stream or too large for memory             -> §8 Reservoir Sampling

What is your inferential goal?
+-- CI for a standard statistic (mean, proportion)       -> §1 Normal / t CI
+-- CI for any statistic, unknown distribution           -> §5 Bootstrap (BCa)
+-- Bias correction for a smooth statistic               -> §6 Jackknife
+-- Population-representative estimate from survey       -> §7 Weighted mean CI
```

**Full comparison table:**

| Method | Replacement | Population structure needed | CI approach | Handles non-smooth stats? | Cost |
| :--- | :--- | :--- | :--- | :--- | :--- |
| SRS | Optional | No | t-interval | Yes (approx) | Low |
| Stratified | No | Yes -- strata | Weighted stratum variance | Yes | Medium |
| Systematic | No | No (ordering only) | ~SRS | Yes (approx) | Low |
| Cluster | No | Yes -- clusters | Between-cluster variance | Yes | Low (field) |
| Bootstrap | Yes | No | Percentile / Basic / BCa | Yes | High -- O(B*n) |
| Jackknife | No | No | Jackknife SE + t | No -- smooth only | Medium -- O(n) |
| Weighted | Optional | No | Weighted SE + t | Yes | Low-Medium |
| Reservoir | No | No | SRS after fill | Yes | Single-pass |